In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Energy Signature

```{admonition} Goal
:class: tip
Plot an energy signature showing heating demand as a function of outdoor temperature. This reveals the building's heating behavior and the balance point temperature.
```

```{admonition} Data Basis
:class: note
Building energy signature data (`bldg_engy_sig_simple.csv`) bundled with the `pyedautils` package. Contains columns `timestamp`, `temperature`, and `value` (daily heating energy).
```

## Data Preparation

In [ ]:
import numpy as np
import pandas as pd
from importlib import resources

data_path = resources.files("pyedautils") / "data" / "bldg_engy_sig_simple.csv"
df = pd.read_csv(data_path, sep=";")

def remove_outliers_by_temp_bin(df, temp_col, value_col, bin_width=2,
                                lower_q=0.01, upper_q=0.99):
    """Remove rows where value_col falls outside the per-temperature-bin quantiles."""
    df = df.copy()
    df["_bin"] = pd.cut(df[temp_col], bins=np.arange(
        df[temp_col].min() - bin_width,
        df[temp_col].max() + bin_width * 2,
        bin_width,
    ))
    bounds = df.groupby("_bin", observed=True)[value_col].quantile(
        [lower_q, upper_q]
    ).unstack()
    bounds.columns = ["lo", "hi"]
    df = df.join(bounds, on="_bin")
    mask = df[value_col].between(df["lo"], df["hi"])
    print(f"Removed {(~mask).sum()} outliers out of {len(df)} rows")
    return df.loc[mask].drop(columns=["_bin", "lo", "hi"])

df = remove_outliers_by_temp_bin(df, "temperature", "value")
df.head()

## Visualization

In [ ]:
from pyedautils.plots.energy import plot_energy_signature

fig = plot_energy_signature(
    data=df,
    title="Building Energy Signature",
    xlab="Outside Temperature [°C]",
    ylab="Heating Energy [kWh/d]",
)
fig.show()

```{dropdown} Customization
You can adjust the axis labels and title. To analyze a different time period, filter the DataFrame before plotting.

```python
df_winter = df[df["timestamp"].dt.month.isin([11, 12, 1, 2, 3])]
fig = plot_energy_signature(df_winter, title="Energy Signature — Winter Only")
```
```